In [ ]:
# Google Colab setup
!pip install -q kailash-ml polars plotly gdown python-dotenv

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# ASCENT09 — Exercise 8: Capstone — Agent Deployment via Nexus
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Deploy a complete agent system — RAG + ReActAgent + MCP
#   tools — as a multi-channel service via Nexus (API + CLI + MCP).
#
# TASKS:
#   1. Build RAG-enhanced agent
#   2. Configure Nexus for multi-channel
#   3. Deploy as API endpoint
#   4. Deploy as CLI interface
#   5. Test across channels with session persistence
#   6. Measure latency and cost per query
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import math
import os
import time

import polars as pl
from dotenv import load_dotenv

from kaizen import InputField, OutputField, Signature
from kaizen.core import BaseAgent
from kaizen_agents import Delegate
from kailash_nexus import Nexus

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()

model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))



## TASK 1: Build RAG-enhanced agent


Retrieve top-k relevant documents for a query.


Answer questions using retrieved regulatory context.


In [ ]:
loader = ASCENTDataLoader()
regulations = loader.load("ascent09", "sg_regulations.parquet")

print(f"=== Singapore Regulations: {regulations.height} documents ===")


# Simple vector store for RAG
def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b + 1e-10)


# Build TF-IDF index for retrieval
import re
from collections import Counter

all_docs = regulations["text"].to_list()
doc_tokens = []
vocab_freq: Counter = Counter()

for doc in all_docs:
    tokens = re.sub(r"[^a-z0-9\s]", " ", doc.lower()).split()
    doc_tokens.append(tokens)
    vocab_freq.update(set(tokens))

vocab = [t for t, f in vocab_freq.most_common(2000) if f > 1]
token_idx = {t: i for i, t in enumerate(vocab)}
n_docs = len(all_docs)

# TF-IDF vectors
idf = {t: math.log(n_docs / (1 + vocab_freq[t])) for t in vocab}

doc_vectors = []
for tokens in doc_tokens:
    tf = Counter(tokens)
    total = len(tokens) if tokens else 1
    vec = [0.0] * len(vocab)
    for t, count in tf.items():
        if t in token_idx:
            vec[token_idx[t]] = (count / total) * idf.get(t, 0.0)
    doc_vectors.append(vec)


def retrieve(query: str, top_k: int = 3) -> list[str]:
    q_tokens = re.sub(r"[^a-z0-9\s]", " ", query.lower()).split()
    q_tf = Counter(q_tokens)
    q_total = len(q_tokens) if q_tokens else 1
    q_vec = [0.0] * len(vocab)
    for t, count in q_tf.items():
        if t in token_idx:
            q_vec[token_idx[t]] = (count / q_total) * idf.get(t, 0.0)

    scores = [(i, cosine_similarity(q_vec, dv)) for i, dv in enumerate(doc_vectors)]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [all_docs[i][:500] for i, _ in scores[:top_k]]


# Test retrieval
test_results = retrieve("AI governance regulation Singapore")
print(f"\nRAG retrieval test: {len(test_results)} documents retrieved")
print(f"Top result (first 100 chars): {test_results[0][:100]}...")


class RAGSignature(Signature):

    question: str = InputField(description="User's question about regulations")
    context: str = InputField(description="Retrieved regulatory documents")
    # TODO: Define OutputField for answer (based on context)
    answer: str = ____
    # TODO: Define OutputField for sources (list of source references)
    sources: list[str] = ____
    # TODO: Define OutputField for confidence (score 0-1)
    confidence: float = ____


class RAGAgent(BaseAgent):
    signature = RAGSignature
    model = os.environ.get("DEFAULT_LLM_MODEL")
    max_llm_cost_usd = 2.0


rag_agent = RAGAgent()

print(f"\n=== RAG Agent Built ===")
print(f"Retriever: TF-IDF over {len(all_docs)} regulation documents")
print(f"Generator: BaseAgent with RAGSignature")



## TASK 2: Configure Nexus for multi-channel


Handle a regulatory query through the RAG pipeline.


In [ ]:
async def handle_query(query: str) -> dict:
    # Retrieve relevant context
    contexts = retrieve(query, top_k=3)
    context_str = "\n\n---\n\n".join(contexts)

    # Generate answer
    result = await rag_agent.run(
        question=query,
        context=context_str,
    )

    return {
        "answer": result.answer,
        "sources": result.sources,
        "confidence": result.confidence,
    }


# TODO: Create a Nexus instance and register the handle_query handler
app = ____
app.register(handle_query)

print(f"\n=== Nexus Configuration ===")
print(f"Nexus deploys the same handler across three channels:")
print(f"  - API: REST endpoint at /query")
print(f"  - CLI: Interactive command-line interface")
print(f"  - MCP: Tool accessible by other agents")
print(f"Zero code changes between channels — Nexus handles transport.")



## TASK 3: Deploy as API endpoint


In [ ]:
async def test_api_channel():
    print(f"\n=== API Channel Test ===")
    # TODO: Create a Nexus session for state persistence
    session = ____

    queries = [
        "What are Singapore's AI governance requirements for financial services?",
        "How does the EU AI Act classify high-risk AI systems?",
        "What is the timeline for AI Act compliance?",
    ]

    for q in queries:
        start = time.time()
        result = await handle_query(q)
        latency = time.time() - start

        print(f"\nQ: {q}")
        print(f"A: {result['answer'][:200]}...")
        print(f"Sources: {result['sources'][:2]}")
        print(f"Confidence: {result['confidence']}")
        print(f"Latency: {latency:.2f}s")

    return session


session  = await test_api_channel()



## TASK 4: Deploy as CLI interface


In [ ]:
print(f"\n=== CLI Channel ===")
print(f"In production, Nexus provides an interactive CLI:")
print(f"  $ kailash nexus cli --app regulatory-rag")
print(f"  > What AI regulations apply to my healthcare startup?")
print(f"  [answer with sources and confidence]")
print(f"Same handler, same session state — different transport.")



## TASK 5: Session persistence across channels


In [ ]:
print(f"\n=== Session Persistence ===")
print(f"Session ID: {session}")
print(f"Nexus sessions persist state across channels:")
print(f"  1. User asks via API: 'What are Singapore AI rules?'")
print(f"  2. Same user follows up via CLI: 'What about financial services?'")
print(f"  3. Session context preserved — agent remembers the first query")
print(f"This is critical for conversational RAG where context builds up.")



## TASK 6: Measure latency and cost


In [ ]:
async def benchmark():
    print(f"\n=== Performance Benchmark ===")

    n_queries = 5
    latencies = []
    queries = [
        "What is MAS TRM?",
        "Explain AI Verify framework",
        "Singapore data protection for AI",
        "Responsible AI principles",
        "AI governance best practices",
    ]

    for q in queries:
        start = time.time()
        await handle_query(q)
        latencies.append(time.time() - start)

    avg_latency = sum(latencies) / len(latencies)
    p95_latency = sorted(latencies)[int(0.95 * len(latencies))]

    print(f"Queries: {n_queries}")
    print(f"Avg latency: {avg_latency:.2f}s")
    print(f"P95 latency: {p95_latency:.2f}s")
    print(f"Estimated cost per query: $0.002-0.01 (depends on model)")

    return latencies


latencies  = await benchmark()

print(f"\n=== Capstone Summary ===")
print(f"Built a complete agent system:")
print(f"  1. RAG: TF-IDF retriever + BaseAgent generator")
print(f"  2. Nexus: multi-channel deployment (API + CLI + MCP)")
print(f"  3. Sessions: persistent state across channels")
print(f"  4. Benchmarking: latency and cost tracking")
print(f"This is the Kailash agent lifecycle — from documents to production.")

print("\n✓ Exercise 8 complete — RAG agent deployed via Nexus multi-channel")

